# render_table: разбор

Задача из `0.02.ControlFlow/tasks/render_table`: вывести таблицу колонками,
как это делают `ls -l`, `docker ps` и `git log --oneline`. На входе — список
строк, каждая строка — список ячеек-строк, первая строка заголовок; вторым
аргументом — выравнивание, по символу на колонку. На выходе — список готовых
строк вывода.

Целевой вид из README — на него будем равняться весь разбор:

```
имя   | балл
------+-----
Алиса |   10
Боб   |    9
```

Разбор устроен так: сначала сам инструмент (спецификация формата), потом
решение по шагам, потом проверка — публичные тесты и сверка двух независимых
реализаций, включая решение студента. В конце — короткая викторина.

---

## Часть 1. Инструмент: спецификация формата

### 1.1. Три выравнивания

Символ выравнивания ставится в спецификации после двоеточия, перед шириной
поля. Строка дополняется пробелами до ширины. Вертикальная черта справа —
маячок, чтобы видеть края поля:

In [1]:
print(f"{'ab':<6}" + "|")
print(f"{'ab':^6}" + "|")
print(f"{'ab':>6}" + "|")

ab    |
  ab  |
    ab|


### 1.2. Центрирование с нечётным остатком

Если свободных позиций нечётное число, поровну их не разделить. Лишний пробел
уходит **вправо** — так ведёт себя `^` в Python, и это зафиксировано в README:

In [2]:
print(repr(f"{'ab':^5}"))    # свободных 3: один влево, два вправо
print(repr(f"{'abc':^6}"))   # свободных 3: полтора+полтора не бывает

' ab  '
' abc  '


### 1.3. Ширина меньше содержимого

Ничего не обрезается: строка выводится целиком, поле оказывается шире
заданного. Именно поэтому ширину колонки надо считать по самой длинной ячейке —
иначе таблица разъедется:

In [3]:
f"{'abcdef':<3}"

'abcdef'

### 1.4. Умолчание зависит от типа

Без символа выравнивания строки прижимаются влево, а числа — вправо. В задаче
выравнивание приходит аргументом, поэтому полагаться на умолчание нельзя:

In [4]:
print(repr(f"{'x':6}"), repr(f"{42:6}"))

'x     ' '    42'


### 1.5. Ключ к задаче: вложенные поля подстановки

Спецификация `f"{value:<12}"` записана литералом — а ширина колонки станет
известна только после просмотра всей таблицы. Спецификация не обязана быть
литералом: внутри неё разрешены **вложенные поля подстановки**, и они
вычисляются до того, как формат применяется к значению:

In [5]:
align, width = "^", 5          # станут известны в рантайме
f"{'ab':{align}{width}}"

' ab  '

То есть в спецификацию подставляются и выравнивание, и ширина:
`f"{cell:{alignments[i]}{widths[i]}}"`. Это весь «секрет» задачи — дальше
обычная аккуратная сборка строк.

### 1.6. Символы, а не байты

Ширина считается в символах. Кириллическая буква занимает два байта в UTF-8,
но выравнивается как один знак — как и должно быть для текста на экране:

In [6]:
word = "абв"
print(len(word), len(word.encode("utf-8")))
print(repr(f"{word:^7}"))

3 6
'  абв  '


---

## Часть 2. Собираем решение по шагам

Договоримся о терминах: **ширина колонки** — длина самой длинной ячейки
в этой колонке, считая заголовок; **строка вывода** — ячейки, расставленные
по своим ширинам и склеенные через ` | `.

### 2.1. Ширины колонок

Прямое прочтение требования: по каждой колонке взять максимум по всем строкам
таблицы:

In [7]:
rows = [["имя", "балл"], ["Алиса", "10"], ["Боб", "9"]]

widths = [max(len(row[c]) for row in rows) for c in range(len(rows[0]))]
widths

[5, 4]

Получили `[5, 4]`: `"Алиса"` длиннее заголовка, `"балл"` длиннее `"10"`.

### 2.2. Строка таблицы

Каждую ячейку форматируем её выравниванием и её шириной, склеиваем через
разделитель, хвостовые пробелы конца строки срезаем:

In [8]:
def line(cells: list[str], alignments: str, widths: list[int]) -> str:
    formatted = (f"{cell:{a}{w}}" for cell, a, w in zip(cells, alignments, widths))
    return " | ".join(formatted).rstrip()


print(line(rows[1], "<>", widths))
print(repr(line(rows[1], "<>", widths)))

Алиса |   10
'Алиса |   10'


`rstrip` трогает только конец строки: внутренние пробелы разделителя целы,
а ведущие пробелы (при `>`) остаются частью строки — README это отдельно
оговаривает. Ровно то, что требуется.

### 2.3. Разделитель

Под каждой колонкой дефисов столько, сколько её ширина; на месте разделителя
ячеек — `-+-`. Подводный камень чтения: дефисы `-+-` сливаются с дефисами
соседних колонок, и при взгляде глазами границы колонок в этой строке
не угадать:

In [9]:
"-+-".join("-" * w for w in widths)

'------+-----'

In [10]:
print("--" + "-+-" + "--")              # ширины 2 и 2
print("-+-".join(["--", "---", "-"]))   # ширины 2, 3, 1

---+---
---+-----+--


Ширины 2 и 2 дают `---+---` — по три дефиса с каждой стороны, хотя ни одна
колонка не шире двух. Считать эти строки глазами — верный способ ошибиться;
ниже в разборе это ещё встретится.

### 2.4. Одна колонка — не особый случай

У одиночного элемента `join` нет пар, разделитель не вставляется, а `-+-`
ни разу не вызывается. Отдельная ветка `if` для одноколоночной таблицы
не нужна:

In [11]:
print("-+-".join(["-"]))
line(["a"], "<", [1])

-


'a'

### 2.5. Всё вместе

Собираем: пустая таблица — пустой список; ширины; строка-заголовок;
разделитель; остальные строки:

In [12]:
def render_table(rows: list[list[str]], alignments: str) -> list[str]:
    if not rows:
        return []
    widths = [max(len(row[c]) for row in rows) for c in range(len(rows[0]))]

    def line(cells: list[str]) -> str:
        formatted = (
            f"{cell:{align}{width}}"
            for cell, align, width in zip(cells, alignments, widths)
        )
        return " | ".join(formatted).rstrip()

    separator = "-+-".join("-" * width for width in widths)
    return [line(rows[0]), separator] + [line(row) for row in rows[1:]]


for row in render_table([["имя", "балл"], ["Алиса", "10"], ["Боб", "9"]], "<>"):
    print(row)

имя   | балл
------+-----
Алиса |   10
Боб   |    9


In [13]:
render_table([["a"]], "<"), render_table([], "<")

(['a', '-'], [])

Оба края из README на месте: одна колонка даёт `['a', '-']`, пустая таблица —
`[]`.

---

## Часть 3. Проверка

### 3.1. Публичные тесты

Три проверки из условия:

In [14]:
assert render_table(
    [["имя", "балл"], ["Алиса", "10"], ["Боб", "9"]], "<>"
) == [
    "имя   | балл",
    "------+-----",
    "Алиса |   10",
    "Боб   |    9",
]
assert render_table([["a"]], "<") == ["a", "-"]
assert render_table([], "<") == []
"3 из 3"

'3 из 3'

### 3.2. Вторая реализация: решение студента

Проверенное решение студента — верное: проходит публичные тесты, mypy чисто,
и на 2000 случайных таблицах совпадает с эталоном дословно. Код дословно,
переименована только сама функция — чтобы две реализации жили в одном
ноутбуке:

In [15]:
def render_table_student(rows: list[list[str]], alignments: str) -> list[str]:
    if len(rows) == 0:
        return []

    count: list[int] = []
    for i in rows:
        if not count:
            for j in range(len(i)):
                count.append(len(i[j]))
        else:
            for j in range(len(i)):
                count[j] = max(count[j], len(i[j]))

    alignments_end: list[str] = []
    if len(alignments) > 1:
        for al in alignments:
            alignments_end.append(al)
    else:
        for _al2 in range(len(count)):
            alignments_end.append(alignments)
    otv = []
    if len(count) > 1:
        hate_ruff = ""
        zag = ""
        for first in range(len(count) - 1):
            hate_ruff += f"{rows[0][first]:{alignments_end[first]}{count[first]}}" + " | "
            zag += count[first] * "-" + "-+-"
        hate_ruff += f"{rows[0][-1]:{alignments_end[-1]}{count[-1]}}"
        hate_ruff = hate_ruff.rstrip()
        zag += count[-1] * "-"
    else:
        hate_ruff = f"{rows[0][0]:{alignments_end[0]}{count[0]}}".rstrip()
        zag = count[-1] * "-"
    otv.append(hate_ruff)
    otv.append(zag)
    if len(rows) > 1:
        for x in range(1, len(rows)):
            if len(count) > 1:
                hate_ruff_rep = ""
                for second in range(len(count) - 1):
                    hate_ruff_rep += f"{rows[x][second]:{alignments_end[second]}{count[second]}}"
                    hate_ruff_rep += " | "
                hate_ruff_rep += f"{rows[x][-1]:{alignments_end[-1]}{count[-1]}}"
                hate_ruff_rep = hate_ruff_rep.rstrip()
            else:
                hate_ruff_rep = f"{rows[x][0]:{alignments_end[0]}{count[0]}}".rstrip()
            otv.append(hate_ruff_rep)
    return otv

Как оно устроено, по блокам:

* **Ширины** (`count`): первая строка инициализирует список длин, остальные
  обновляют максимумы. Работает верно; хитрость `if not count` полагается
  на то, что первая строка не пуста — по контракту таблицы так и есть.
* **Выравнивания** (`alignments_end`): символы разбираются в список, одиночный
  символ размножается на все колонки. Контракт README гарантирует символ на
  колонку, так что вторая ветка — запасливость сверх условий задачи.
* **Сборка**: четыре ветки — заголовок/данные × колонок больше одной/ровно
  одна. В каждой ветке ячейки дописываются к строке с `" | "`, потом `rstrip`.
  Ветка одноколоночной таблицы здесь не нужна (см. 2.4), но и не вредит:
  получается тот же результат.

Что стоило бы подтянуть — на корректность не влияет, но видно на ревью:

* `ruff` просит заменить цикл с `append` на `list(alignments)` (правило
  PERF402) — копирование итерируемого в список — это ровно оно;
* имена `otv`, `zag`, `count`, `first`/`second` держат читателя в напряжении:
  `count` — это ширины, `zag` — разделитель, `first` — индекс колонки;
* четыре ветки сборки — дублирование: одна функция `line` из 2.2 закрывает
  все случаи, и разбор «какая из четырёх веток отвечает за эту строку»
  исчезает вместе с ветками.

### 3.3. Сверка двух решений

Приём для задач с дословным выводом: пишутся две независимые реализации,
и на случайных входах обязаны совпадать. Расхождения ловят обе — и неверно
понятый контракт, и опечатку:

In [16]:
from random import Random

rng = Random(2024)
alphabet = "абвАБВxyz019 ."
checked = mismatches = 0

for _ in range(2000):
    n_cols = rng.randint(1, 5)
    n_rows = rng.randint(1, 6)
    table = [
        ["".join(rng.choice(alphabet) for _ in range(rng.randint(0, 8)))
         for _ in range(n_cols)]
        for _ in range(n_rows)
    ]
    alignments = "".join(rng.choice("<^>") for _ in range(n_cols))
    checked += 1
    if render_table(table, alignments) != render_table_student(table, alignments):
        mismatches += 1

print(f"таблиц: {checked}, расхождений: {mismatches}")

таблиц: 2000, расхождений: 0


Ноль расхождений на 2000 таблицах с кириллицей, пробелами и пустыми ячейками.

Почему «сверять код с кодом», а не глазами: при проверке этой задачи
ревьюер **шесть раз** посчитал ожидаемые строки руками неверно — то центр
с нечётным остатком, то слияние дефисов в разделителе — и во всех шести
спорах прав оказался код, что и подтвердила сверка. Человек читает
выровненные таблицы плохо; две независимые программы — надёжно.

---

## Часть 4. Викторина «предскажи вывод»

### Вопрос 1

In [17]:
print(repr(f"{'ab':^5}"))

' ab  '


**Разбор.** `' ab  '`. Свободных позиций три — нечётно: один пробел влево,
два вправо. Лишний всегда уходит вправо.

### Вопрос 2

In [18]:
print("-+-".join(["--", "---", "-"]))

---+-----+--


**Разбор.** `---+-----+--`. Ширины колонок 2, 3, 1 — но читать ответ надо
помня о слиянии: дефисы `-+-` примыкают к дефисам колонок, и счёт по выводу
не совпадает со счётом по ширинам.

### Вопрос 3

In [19]:
print(repr(f"{'abcdef':<3}".rstrip()))

'abcdef'


**Разбор.** `'abcdef'`. Поле уже ширины спецификации не режет — строка
выводится целиком, и `rstrip` резать нечего.

### Вопрос 4

In [20]:
print(repr(f"{'ab':^5}".rstrip()))

' ab'


**Разбор.** `' ab'`. Хвостовые пробелы срезаны, ведущий остался: он — часть
выравнивания, а не мусор. Поэтому `rstrip` в задаче применяется к концу
строки вывода, и только к нему.

### Вопрос 5

In [21]:
print(repr(f"{42:6}"), repr(f"{'x':6}"))

'    42' 'x     '


**Разбор.** `'    42'` и `'x     '`. Умолчание зависит от типа: числа
вправо, строки влево. В `render_table` выравнивание приходит аргументом —
значит, символ в спецификации ставится явно, на умолчание не полагаемся.

---

## Итог

Чеклист решения `render_table`:

1. Ширина колонки — максимум по колонке, включая заголовок.
2. Спецификация формата собирается в рантайме: `f"{cell:{align}{width}}"` —
   вложенные поля подстановки.
3. Строка — `" | ".join(...)` по отформатированным ячейкам, `rstrip` конца.
4. Разделитель — `"-+-".join("-" * w ...)`: одна формула, отдельной ветки
   для одной колонки не нужно.
5. Пустая таблица — `[]`, до всех вычислений.
6. Проверять — двумя независимыми реализациями на случайных входах:
   руки таблицу считают плохо, код — хорошо.